# Tensor Decomposition Experiments
*Thomas Dooms applicant task - extended experiment notebook*

This notebook turns the provided tensor-decomposition skeleton into a small experimental study. The goal is not merely to reconstruct the bilinear interaction tensor, but to find shared components that are compact, faithful, stable, and easy to name as digit strokes or counter-strokes.

## What We Are Testing

The trained bilinear MNIST model computes logits from a third-order interaction tensor:

$$\text{logit}_c(x) = \sum_{i,j} B_{c,i,j}\,x_i\,x_j$$

The tutorial baseline fits a shared CP-style factorization:

$$B_{c,i,j} \approx \sum_r L_{i,r} R_{j,r} D_{c,r}$$

That already shares features across classes, but pure reconstruction pressure can still produce noisy or superposed components. Below we compare several priors:

- the provided sparse baseline
- symmetric factors: $L=R$
- sparse and smooth image-domain factors
- direct positive/negative evidence splits
- nonnegative stroke factors
- eigenvector-seeded shared dictionaries
- multi-seed consensus components

The core question is whether a small loss in reconstruction quality buys a large gain in interpretability.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import math
import os
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm import tqdm

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise
from einops import einsum

try:
    import pandas as pd
except Exception:
    pd = None

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
EXPORT_FIGURES = True

if torch.cuda.is_available():
    device = "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

RUN_PROFILE = os.environ.get("RUN_PROFILE", "balanced")
if RUN_PROFILE == "full":
    MODEL_EPOCHS = 20
    BASE_RANK = 64
    LARGE_RANK = 96
    BASELINE_STEPS = 200
    VARIANT_STEP_SCALE = 1.0
    CONSENSUS_STEPS = 300
else:
    MODEL_EPOCHS = 4
    BASE_RANK = 32
    LARGE_RANK = 48
    BASELINE_STEPS = 80
    VARIANT_STEP_SCALE = 0.22
    CONSENSUS_STEPS = 80

torch.set_float32_matmul_precision("high") if hasattr(torch, "set_float32_matmul_precision") else None

def set_seed(seed=0):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)
device

'mps'

In [2]:
def save_fig(fig, name):
    """Save interactive HTML and, when kaleido is available, a README-friendly PNG."""
    if not EXPORT_FIGURES:
        return fig
    html_path = FIG_DIR / f"{name}.html"
    png_path = FIG_DIR / f"{name}.png"
    fig.write_html(str(html_path), include_plotlyjs="cdn")
    try:
        fig.write_image(str(png_path), scale=2)
    except Exception as exc:
        print(f"Saved {html_path}; PNG export skipped ({type(exc).__name__}: {exc})")
    return fig


def show_table(rows, sort_by=None):
    if pd is not None:
        df = pd.DataFrame(rows)
        if sort_by is not None and sort_by in df:
            df = df.sort_values(sort_by, ascending=False)
        display(df)
        return df
    for row in rows:
        print(row)
    return rows


def make_optimizer(params, lr=2e-2, momentum=0.95, weight_decay=0.0):
    # Muon was used in the skeleton. Fall back cleanly for environments with older torch.
    if hasattr(torch.optim, "Muon") and momentum is not None:
        return torch.optim.Muon(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

## Train The Original Bilinear Model

In [3]:
set_seed(0)
model = Model.from_config(epochs=MODEL_EPOCHS).to(device)
metrics = model.fit(train, test, RandomGaussianNoise(std=0.4).to(device))

with torch.no_grad():
    original_acc = (model(test.x).argmax(-1) == test.y).float().mean().item()
original_acc

epoch 01/4 loss=0.4234 acc=0.9587


epoch 02/4 loss=0.2086 acc=0.9647


epoch 03/4 loss=0.1866 acc=0.9672


epoch 04/4 loss=0.1648 acc=0.9740


0.9739999771118164

## Build The Interaction Tensor

The decomposition variants below operate in pixel/input space, so the original bilinear tensor includes the image embedding matrix. This makes each learned component directly viewable as a `28 x 28` pattern.

In [4]:
@torch.no_grad()
def interaction_tensor(model):
    """Return the symmetrized input-space tensor B[c, pix_i, pix_j]."""
    l, r = model.w_lr[0].unbind()
    b = einsum(
        model.w_u,
        l,
        r,
        model.w_e,
        model.w_e,
        "cls out, out hid1, out hid2, hid1 pix1, hid2 pix2 -> cls pix1 pix2",
    )
    return 0.5 * (b + b.mT)

B = interaction_tensor(model).detach()
B_norm = B.norm()
B.shape, B_norm.item()

(torch.Size([10, 784, 784]), 10.269742965698242)

## Evaluation Helpers

In [5]:
def cosine_to_target(approx, target=B):
    return F.cosine_similarity(approx.flatten(), target.flatten(), dim=0)


def normalize_columns(x, eps=1e-8):
    return x / x.norm(dim=0, keepdim=True).clamp_min(eps)


def gini_1d(x, eps=1e-8):
    x = x.detach().abs().flatten()
    if x.numel() == 0 or x.sum() < eps:
        return torch.tensor(0.0, device=x.device)
    x = x.sort().values
    n = x.numel()
    idx = torch.arange(1, n + 1, device=x.device, dtype=x.dtype)
    return ((2 * idx - n - 1) * x).sum() / (n * x.sum().clamp_min(eps))


def mean_gini(patterns):
    cols = patterns.T if patterns.shape[0] == 784 else patterns
    return torch.stack([gini_1d(col) for col in cols]).mean()


def total_variation(patterns):
    imgs = patterns.T.reshape(-1, 1, 28, 28)
    dy = (imgs[:, :, 1:, :] - imgs[:, :, :-1, :]).abs().mean()
    dx = (imgs[:, :, :, 1:] - imgs[:, :, :, :-1]).abs().mean()
    return dx + dy


def patch_locality(patterns, patch=7, eps=1e-8):
    """Mean fraction of each component's squared mass captured by its best local patch."""
    imgs = patterns.T.reshape(-1, 1, 28, 28)
    energy = imgs.square()
    pooled = F.avg_pool2d(energy, kernel_size=patch, stride=1) * (patch * patch)
    best = pooled.flatten(1).max(dim=1).values
    total = energy.flatten(1).sum(dim=1).clamp_min(eps)
    return (best / total).mean()


def class_selectivity(down, eps=1e-8):
    """1 - normalized entropy of abs class weights. Higher means fewer classes per component."""
    p = down.detach().abs()
    p = p / p.sum(dim=0, keepdim=True).clamp_min(eps)
    entropy = -(p * (p + eps).log()).sum(dim=0) / math.log(p.shape[0])
    return 1 - entropy.mean()


def metric_dict(name, factor_model, target=B):
    with torch.no_grad():
        approx = factor_model.tensor()
        logits = factor_model(test.x)
        plus, minus, down, sigma = factor_model.decompose()
        patterns = torch.cat([plus, minus], dim=1)
        return {
            "name": name,
            "similarity": cosine_to_target(approx, target).item(),
            "test_acc": (logits.argmax(-1) == test.y).float().mean().item(),
            "pattern_gini": mean_gini(patterns).item(),
            "locality_7x7": patch_locality(patterns).item(),
            "class_selectivity": class_selectivity(down).item(),
            "top_sigma_frac": (sigma[:8].sum() / sigma.sum().clamp_min(1e-8)).item(),
        }


def builtin_sparse_metrics(name, sparse, original_model=model):
    with torch.no_grad():
        plus, minus, down, sigma = sparse.decompose()
        patterns = torch.cat([plus, minus], dim=1)
        return {
            "name": name,
            "similarity": sparse.similarity(original_model).item(),
            "test_acc": (sparse(test.x).argmax(-1) == test.y).float().mean().item(),
            "pattern_gini": mean_gini(patterns).item(),
            "locality_7x7": patch_locality(patterns).item(),
            "class_selectivity": class_selectivity(down).item(),
            "top_sigma_frac": (sigma[:8].sum() / sigma.sum().clamp_min(1e-8)).item(),
        }

## Plotting Helpers

In [6]:
def component_figure(plus, minus, down, sigma=None, k=8, title="components"):
    plus, minus, down = plus.detach().cpu(), minus.detach().cpu(), down.detach().cpu()
    k = min(k, plus.shape[1])
    fig = make_subplots(
        rows=3,
        cols=k,
        row_titles=["positive", "negative", "logits"],
        vertical_spacing=0.08,
        horizontal_spacing=0.02,
    )
    for i in range(k):
        params = dict(showscale=False, colorscale="RdBu", zmid=0)
        fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i + 1)
        fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i + 1)
        colors = ["#666666"] * 10
        top_cls = int(down[:, i].abs().argmax())
        colors[top_cls] = "#c43c39"
        fig.add_bar(y=down[:, i], marker_color=colors, showlegend=False, row=3, col=i + 1)
    fig.update_xaxes(visible=False).update_yaxes(visible=False)
    fig.update_layout(
        width=max(900, k * 125),
        height=420,
        title=title,
        margin=dict(l=50, r=5, b=5, t=45),
        template="plotly_white",
    )
    return fig


def plot_factor_components(factor_model, name, k=8):
    plus, minus, down, sigma = factor_model.decompose()
    fig = component_figure(plus, minus, down, sigma, k=k, title=name)
    return save_fig(fig, f"components_{name.replace(' ', '_').lower()}")


def plot_sigma(sigma, name):
    fig = px.bar(
        sigma.detach().cpu(),
        template="plotly_white",
        labels=dict(index="component", value="sigma"),
        title=f"Component spectrum: {name}",
    )
    return save_fig(fig, f"sigma_{name.replace(' ', '_').lower()}")

def component_strengths(factor_model):
    if hasattr(factor_model, "component_strengths"):
        try:
            return factor_model.component_strengths()
        except NotImplementedError:
            pass
    return factor_model.decompose()[-1]

## Baseline: Provided Sparse Model

In [7]:
def fit_builtin_sparse(rank=64, steps=200, lr=0.02, seed=0):
    set_seed(seed)
    sparse = Sparse.from_config(rank=rank).to(device)
    optimizer = make_optimizer(sparse.parameters(), lr=lr, momentum=0.95)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)

    torch.set_grad_enabled(True)
    for _ in (pbar := tqdm(range(steps), desc="builtin sparse")):
        loss = 1 - sparse.similarity(model)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()
        pbar.set_description(f"builtin sparse loss={loss.item():.4f}")
    torch.set_grad_enabled(False)
    return sparse

builtin = fit_builtin_sparse(rank=BASE_RANK, steps=BASELINE_STEPS, seed=0)
builtin_row = builtin_sparse_metrics(f"provided Sparse r{BASE_RANK}", builtin)
show_table([builtin_row])

builtin sparse:   0%|          | 0/80 [00:00<?, ?it/s]

builtin sparse loss=0.9999:   0%|          | 0/80 [00:00<?, ?it/s]

builtin sparse loss=0.9727:   0%|          | 0/80 [00:00<?, ?it/s]

builtin sparse loss=0.9727:   2%|▎         | 2/80 [00:00<00:04, 16.22it/s]

builtin sparse loss=0.9219:   2%|▎         | 2/80 [00:00<00:04, 16.22it/s]

...

builtin sparse loss=0.1411:  98%|█████████▊| 78/80 [00:02<00:00, 32.99it/s]

builtin sparse loss=0.1411:  98%|█████████▊| 78/80 [00:02<00:00, 32.99it/s]

builtin sparse loss=0.1411: 100%|██████████| 80/80 [00:02<00:00, 32.40it/s]

,name,similarity,test_acc,pattern_gini,locality_7x7,class_selectivity,top_sigma_frac
0,provided Sparse r32,0.858908,0.9492,0.440377,0.264228,0.106274,0.295426


,name,similarity,test_acc,pattern_gini,locality_7x7,class_selectivity,top_sigma_frac
0,provided Sparse r32,0.858908,0.9492,0.440377,0.264228,0.106274,0.295426


In [8]:
plus, minus, down, sigma = builtin.decompose()
fig = component_figure(plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu(), k=10, title="Provided Sparse baseline")
save_fig(fig, "components_provided_sparse_baseline")

In [9]:
plot_sigma(sigma.cpu(), "provided Sparse baseline")

## Custom Factorized Models

In [10]:
@dataclass
class RegConfig:
    l1: float = 0.0
    tv: float = 0.0
    d_l1: float = 0.0
    symmetry: float = 0.0
    duplicate: float = 0.0


class BaseFactorModel(nn.Module):
    name = "base"

    def tensor(self):
        raise NotImplementedError

    def component_activations(self, x):
        raise NotImplementedError

    def component_strengths(self):
        raise NotImplementedError

    def decompose(self):
        raise NotImplementedError

    def forward(self, x, mask=None):
        acts = self.component_activations(x)
        if mask is not None:
            acts = acts * mask.to(acts.device)
        return acts @ self.down().T

    def down(self):
        return self.raw_down

    def regularizer(self, cfg: RegConfig):
        plus, minus, down, _ = self.decompose()
        reg = plus.new_tensor(0.0)
        if cfg.l1:
            reg = reg + cfg.l1 * (plus.abs().mean() + minus.abs().mean())
        if cfg.tv:
            reg = reg + cfg.tv * (total_variation(plus) + total_variation(minus))
        if cfg.d_l1:
            reg = reg + cfg.d_l1 * down.abs().mean()
        if cfg.duplicate:
            patterns = normalize_columns(torch.cat([plus, minus], dim=1))
            gram = patterns.T @ patterns
            gram = gram - torch.eye(gram.shape[0], device=gram.device, dtype=gram.dtype)
            reg = reg + cfg.duplicate * gram.square().mean()
        return reg


class CPFactorModel(BaseFactorModel):
    name = "cp"

    def __init__(self, rank=64, scale=0.02):
        super().__init__()
        self.raw_left = nn.Parameter(scale * torch.randn(784, rank))
        self.raw_right = nn.Parameter(scale * torch.randn(784, rank))
        self.raw_down = nn.Parameter(scale * torch.randn(10, rank))

    def left(self):
        return self.raw_left

    def right(self):
        return self.raw_right

    def tensor(self):
        t = einsum(self.down(), self.left(), self.right(), "c r, i r, j r -> c i j")
        return 0.5 * (t + t.mT)

    def component_activations(self, x):
        return (x @ self.left()) * (x @ self.right())

    def component_strengths(self):
        return self.left().norm(dim=0) * self.right().norm(dim=0) * self.down().norm(dim=0)

    def decompose(self):
        l, r, d = self.left(), self.right(), self.down()
        plus = normalize_columns(l + r)
        minus = normalize_columns(l - r)
        sigma = self.component_strengths()
        order = sigma.argsort(descending=True)
        return plus[:, order], minus[:, order], d[:, order], sigma[order]

    def regularizer(self, cfg: RegConfig):
        reg = super().regularizer(cfg)
        if cfg.symmetry:
            reg = reg + cfg.symmetry * (self.left() - self.right()).square().mean()
        return reg


class SymmetricFactorModel(BaseFactorModel):
    name = "symmetric"

    def __init__(self, rank=64, scale=0.02, nonnegative=False):
        super().__init__()
        init = scale * torch.randn(784, rank)
        if nonnegative:
            init = init - 4.0
        self.raw_v = nn.Parameter(init)
        self.raw_down = nn.Parameter(scale * torch.randn(10, rank))
        self.nonnegative = nonnegative

    def v(self):
        return F.softplus(self.raw_v) if self.nonnegative else self.raw_v

    def tensor(self):
        return einsum(self.down(), self.v(), self.v(), "c r, i r, j r -> c i j")

    def component_activations(self, x):
        return (x @ self.v()).square()

    def component_strengths(self):
        return self.v().norm(dim=0).square() * self.down().norm(dim=0)

    def decompose(self):
        v, d = self.v(), self.down()
        plus = normalize_columns(v)
        minus = torch.zeros_like(plus)
        sigma = self.component_strengths()
        order = sigma.argsort(descending=True)
        return plus[:, order], minus[:, order], d[:, order], sigma[order]


class EvidenceSplitModel(BaseFactorModel):
    name = "evidence_split"

    def __init__(self, rank=64, scale=0.02, nonnegative=False):
        super().__init__()
        init_pos = scale * torch.randn(784, rank)
        init_neg = scale * torch.randn(784, rank)
        if nonnegative:
            init_pos = init_pos - 4.0
            init_neg = init_neg - 4.0
        self.raw_pos = nn.Parameter(init_pos)
        self.raw_neg = nn.Parameter(init_neg)
        self.raw_down = nn.Parameter(scale * torch.randn(10, rank))
        self.nonnegative = nonnegative

    def pos(self):
        return F.softplus(self.raw_pos) if self.nonnegative else self.raw_pos

    def neg(self):
        return F.softplus(self.raw_neg) if self.nonnegative else self.raw_neg

    def tensor(self):
        d = self.down()
        return einsum(d, self.pos(), self.pos(), "c r, i r, j r -> c i j") - einsum(
            d, self.neg(), self.neg(), "c r, i r, j r -> c i j"
        )

    def component_activations(self, x):
        return (x @ self.pos()).square() - (x @ self.neg()).square()

    def decompose(self):
        p, n, d = self.pos(), self.neg(), self.down()
        plus = normalize_columns(p)
        minus = normalize_columns(n)
        sigma = (p.norm(dim=0).square() + n.norm(dim=0).square()) * d.norm(dim=0)
        order = sigma.argsort(descending=True)
        return plus[:, order], minus[:, order], d[:, order], sigma[order]

In [11]:
def fit_factor_model(
    factor_model,
    name,
    target=B,
    steps=500,
    lr=0.03,
    reg=RegConfig(),
    seed=0,
    progress=True,
):
    if seed is not None:
        set_seed(seed)
    factor_model = factor_model.to(device)
    optimizer = torch.optim.AdamW(factor_model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)
    history = []

    iterator = range(steps)
    if progress:
        iterator = tqdm(iterator, desc=name)

    torch.set_grad_enabled(True)
    for step in iterator:
        approx = factor_model.tensor()
        recon = 1 - cosine_to_target(approx, target)
        penalty = factor_model.regularizer(reg)
        loss = recon + penalty

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()

        if step % max(1, steps // 50) == 0 or step == steps - 1:
            row = {
                "step": step,
                "loss": loss.item(),
                "recon_loss": recon.item(),
                "penalty": penalty.item(),
                "similarity": (1 - recon).item(),
            }
            history.append(row)
            if progress:
                iterator.set_description(f"{name} sim={row['similarity']:.4f}")
    torch.set_grad_enabled(False)
    return factor_model, history


def plot_history(history, name):
    if pd is not None:
        df = pd.DataFrame(history)
        fig = px.line(df, x="step", y=["recon_loss", "penalty"], title=f"Training losses: {name}", template="plotly_white")
    else:
        fig = px.line(history, x="step", y="recon_loss", title=f"Training loss: {name}", template="plotly_white")
    return save_fig(fig, f"history_{name.replace(' ', '_').lower()}")

## Variant Grid

In [12]:
variant_specs = [
    dict(
        name=f"cp r{BASE_RANK} symmetry-soft",
        factory=lambda: CPFactorModel(rank=BASE_RANK),
        steps=int(350 * VARIANT_STEP_SCALE),
        lr=0.03,
        reg=RegConfig(symmetry=0.05),
        seed=1,
    ),
    dict(
        name=f"symmetric r{BASE_RANK}",
        factory=lambda: SymmetricFactorModel(rank=BASE_RANK),
        steps=int(450 * VARIANT_STEP_SCALE),
        lr=0.04,
        reg=RegConfig(),
        seed=2,
    ),
    dict(
        name=f"symmetric sparse smooth r{BASE_RANK}",
        factory=lambda: SymmetricFactorModel(rank=BASE_RANK),
        steps=int(550 * VARIANT_STEP_SCALE),
        lr=0.04,
        reg=RegConfig(l1=2e-4, tv=2e-3, d_l1=2e-4, duplicate=2e-3),
        seed=3,
    ),
    dict(
        name=f"evidence split r{BASE_RANK}",
        factory=lambda: EvidenceSplitModel(rank=BASE_RANK),
        steps=int(550 * VARIANT_STEP_SCALE),
        lr=0.035,
        reg=RegConfig(),
        seed=4,
    ),
    dict(
        name=f"evidence split sparse smooth r{BASE_RANK}",
        factory=lambda: EvidenceSplitModel(rank=BASE_RANK),
        steps=int(650 * VARIANT_STEP_SCALE),
        lr=0.035,
        reg=RegConfig(l1=2e-4, tv=2e-3, d_l1=2e-4, duplicate=1e-3),
        seed=5,
    ),
    dict(
        name=f"nonnegative strokes r{LARGE_RANK}",
        factory=lambda: EvidenceSplitModel(rank=LARGE_RANK, nonnegative=True),
        steps=int(650 * VARIANT_STEP_SCALE),
        lr=0.03,
        reg=RegConfig(l1=3e-4, tv=2e-3, d_l1=2e-4, duplicate=1e-3),
        seed=6,
    ),
]

trained = {f"provided Sparse r{BASE_RANK}": builtin}
histories = {}
rows = [builtin_row]

for spec in variant_specs:
    set_seed(spec["seed"])
    factor, history = fit_factor_model(
        spec["factory"](),
        spec["name"],
        steps=spec["steps"],
        lr=spec["lr"],
        reg=spec["reg"],
        seed=None,
    )
    trained[spec["name"]] = factor
    histories[spec["name"]] = history
    rows.append(metric_dict(spec["name"], factor))

results = show_table(rows, sort_by="similarity")

cp r32 symmetry-soft:   0%|          | 0/77 [00:00<?, ?it/s]

cp r32 symmetry-soft sim=0.0008:   0%|          | 0/77 [00:00<?, ?it/s]

cp r32 symmetry-soft sim=0.0687:   0%|          | 0/77 [00:00<?, ?it/s]

cp r32 symmetry-soft sim=0.3353:   0%|          | 0/77 [00:00<?, ?it/s]

cp r32 symmetry-soft sim=0.3353:   4%|▍         | 3/77 [00:00<00:02, 28.60it/s]

...

nonnegative strokes r48 sim=0.0871:  98%|█████████▊| 140/143 [00:02<00:00, 56.85it/s]

nonnegative strokes r48 sim=0.0871:  98%|█████████▊| 140/143 [00:02<00:00, 56.85it/s]

nonnegative strokes r48 sim=0.0871:  98%|█████████▊| 140/143 [00:02<00:00, 56.85it/s]

nonnegative strokes r48 sim=0.0871: 100%|██████████| 143/143 [00:02<00:00, 53.93it/s]

,name,similarity,test_acc,pattern_gini,locality_7x7,class_selectivity,top_sigma_frac
5,evidence split sparse smooth r32,0.877034,0.9548,0.440257,0.254946,0.120680,0.306341
4,evidence split r32,0.872256,0.9611,0.434809,0.237860,0.109408,0.290891
1,cp r32 symmetry-soft,0.860983,0.9511,0.437123,0.238074,0.103718,0.299290
0,provided Sparse r32,0.858908,0.9492,0.440377,0.264228,0.106274,0.295426
3,symmetric sparse smooth r32,0.787751,0.9316,0.222727,0.138692,0.108916,0.300652
2,symmetric r32,0.778788,0.8867,0.221623,0.139064,0.116112,0.287478
6,nonnegative strokes r48,0.087126,0.4322,0.010952,0.063500,0.103726,0.277420


In [13]:
# Show training curves for the non-baseline variants.
for name, history in histories.items():
    plot_history(history, name)

## Visual Comparison Of All Variants

In [14]:
for name, factor in trained.items():
    if name == "provided Sparse r64":
        plus, minus, down, sigma = factor.decompose()
        fig = component_figure(plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu(), k=10, title=name)
        save_fig(fig, "components_provided_sparse_r64")
    else:
        plot_factor_components(factor, name, k=10)

## Eigenvector-Seeded Shared Dictionary

This is a bridge back to tutorial 1. We take the largest-magnitude per-class eigenvectors and use them as a global initialization for a symmetric shared dictionary, then fine-tune the shared decomposition. If this works, it suggests the tensor dictionary is compressing per-class eigenfeatures into reusable strokes.

In [15]:
@torch.no_grad()
def eigen_seeded_symmetric(target=B, rank=64):
    # MPS currently lacks linalg.eigh; this one-time initialization is fine on CPU.
    target_cpu = target.detach().cpu()
    vals, vecs = torch.linalg.eigh(target_cpu)
    flat = vals.abs().flatten()
    idx = flat.topk(rank).indices
    cls = idx // vals.shape[1]
    comp = idx % vals.shape[1]

    seed_vectors = torch.stack([vecs[c, :, k] for c, k in zip(cls.tolist(), comp.tolist())], dim=1)
    seed_down = torch.zeros(10, rank)
    for r, (c, k) in enumerate(zip(cls.tolist(), comp.tolist())):
        seed_down[c, r] = vals[c, k]

    m = SymmetricFactorModel(rank=rank).to(target.device)
    m.raw_v.copy_(seed_vectors.to(target.device))
    m.raw_down.copy_(seed_down.to(target.device))
    return m

eigen_model = eigen_seeded_symmetric(B, rank=BASE_RANK)
eigen_model, eigen_history = fit_factor_model(
    eigen_model,
    f"eigen seeded symmetric sparse smooth r{BASE_RANK}",
    steps=int(450 * VARIANT_STEP_SCALE),
    lr=0.02,
    reg=RegConfig(l1=1e-4, tv=1e-3, d_l1=1e-4, duplicate=1e-3),
    seed=None,
)
eigen_name = f"eigen seeded symmetric sparse smooth r{BASE_RANK}"
trained[eigen_name] = eigen_model
histories[eigen_name] = eigen_history
rows.append(metric_dict(eigen_name, eigen_model))
results = show_table(rows, sort_by="similarity")
plot_factor_components(eigen_model, eigen_name, k=10)

eigen seeded symmetric sparse smooth r32:   0%|          | 0/99 [00:00<?, ?it/s]

eigen seeded symmetric sparse smooth r32 sim=0.6447:   0%|          | 0/99 [00:00<?, ?it/s]

eigen seeded symmetric sparse smooth r32 sim=0.5352:   0%|          | 0/99 [00:00<?, ?it/s]

eigen seeded symmetric sparse smooth r32 sim=0.6492:   0%|          | 0/99 [00:00<?, ?it/s]

eigen seeded symmetric sparse smooth r32 sim=0.6426:   0%|          | 0/99 [00:00<?, ?it/s]

...

eigen seeded symmetric sparse smooth r32 sim=0.8025:  95%|█████████▍| 94/99 [00:01<00:00, 50.12it/s]

eigen seeded symmetric sparse smooth r32 sim=0.8025:  95%|█████████▍| 94/99 [00:01<00:00, 50.12it/s]

eigen seeded symmetric sparse smooth r32 sim=0.8025:  95%|█████████▍| 94/99 [00:01<00:00, 50.12it/s]

eigen seeded symmetric sparse smooth r32 sim=0.8025: 100%|██████████| 99/99 [00:01<00:00, 49.92it/s]

,name,similarity,test_acc,pattern_gini,locality_7x7,class_selectivity,top_sigma_frac
5,evidence split sparse smooth r32,0.877034,0.9548,0.440257,0.254946,0.120680,0.306341
4,evidence split r32,0.872256,0.9611,0.434809,0.237860,0.109408,0.290891
1,cp r32 symmetry-soft,0.860983,0.9511,0.437123,0.238074,0.103718,0.299290
0,provided Sparse r32,0.858908,0.9492,0.440377,0.264228,0.106274,0.295426
7,eigen seeded symmetric sparse smooth r32,0.802525,0.9265,0.223442,0.151670,0.117094,0.290446
3,symmetric sparse smooth r32,0.787751,0.9316,0.222727,0.138692,0.108916,0.300652
2,symmetric r32,0.778788,0.8867,0.221623,0.139064,0.116112,0.287478
6,nonnegative strokes r48,0.087126,0.4322,0.010952,0.063500,0.103726,0.277420


## Pick A Best Candidate

In [16]:
# The default ranking weights faithfulness and accuracy first, then interpretability proxies.
def score_row(row):
    return (
        4.0 * row["similarity"]
        + 4.0 * row["test_acc"]
        + 0.8 * row["pattern_gini"]
        + 0.8 * row["locality_7x7"]
        + 0.8 * row["class_selectivity"]
    )

ranked = sorted(rows, key=score_row, reverse=True)
best_name = next(row["name"] for row in ranked if row["name"] in trained and not row["name"].startswith("provided Sparse"))
best = trained[best_name]
best_name, ranked[:3]

('evidence split sparse smooth r32',
 [{'name': 'evidence split sparse smooth r32',
   'similarity': 0.8770343065261841,
   'test_acc': 0.9547999501228333,
   'pattern_gini': 0.4402574598789215,
   'locality_7x7': 0.25494641065597534,
   'class_selectivity': 0.12067985534667969,
   'top_sigma_frac': 0.3063405156135559},
  {'name': 'evidence split r32',
   'similarity': 0.8722564578056335,
   'test_acc': 0.9610999822616577,
   'pattern_gini': 0.43480879068374634,
   'locality_7x7': 0.23786047101020813,
   'class_selectivity': 0.10940754413604736,
   'top_sigma_frac': 0.29089149832725525},
  {'name': 'provided Sparse r32',
   'similarity': 0.8589080572128296,
   'test_acc': 0.9491999745368958,
   'pattern_gini': 0.44037699699401855,
   'locality_7x7': 0.2642282247543335,
   'class_selectivity': 0.10627377033233643,
   'top_sigma_frac': 0.29542607069015503}])

In [17]:
plus, minus, down, sigma = best.decompose()
best_fig = component_figure(plus, minus, down, sigma, k=16, title=f"BEST: {best_name}")
save_fig(best_fig, "components_best_decomposition")
plot_sigma(sigma, f"BEST {best_name}")

## Faithfulness: Keep-Only And Ablation Curves

In [18]:
@torch.no_grad()
def faithfulness_curves(factor_model, name, ks=(1, 2, 4, 8, 16, 24, 32, 48, 64), alias=None):
    sigma = component_strengths(factor_model)
    rank = sigma.numel()
    order = sigma.argsort(descending=True)
    rows = []
    for k in ks:
        k = min(k, rank)
        keep = torch.zeros(rank, device=device)
        keep[order[:k]] = 1
        drop = torch.ones(rank, device=device)
        drop[order[:k]] = 0
        keep_logits = factor_model(test.x, mask=keep)
        drop_logits = factor_model(test.x, mask=drop)
        rows.append({
            "k": k,
            "mode": "keep top-k",
            "acc": (keep_logits.argmax(-1) == test.y).float().mean().item(),
        })
        rows.append({
            "k": k,
            "mode": "drop top-k",
            "acc": (drop_logits.argmax(-1) == test.y).float().mean().item(),
        })
    if pd is not None:
        df = pd.DataFrame(rows)
        fig = px.line(df, x="k", y="acc", color="mode", markers=True, template="plotly_white", title=f"Faithfulness curves: {name}")
    else:
        fig = px.line(rows, x="k", y="acc", color="mode", markers=True, template="plotly_white", title=f"Faithfulness curves: {name}")
    save_fig(fig, f"faithfulness_{name.replace(' ', '_').lower()}")
    if alias is not None:
        save_fig(fig, alias)
    return rows

faithfulness = faithfulness_curves(best, best_name, alias="faithfulness_best_decomposition")
show_table(faithfulness)

,k,mode,acc
0,1,keep top-k,0.1863
1,1,drop top-k,0.9393
2,2,keep top-k,0.2231
3,2,drop top-k,0.9322
4,4,keep top-k,0.2723
5,4,drop top-k,0.9251
6,8,keep top-k,0.4145
7,8,drop top-k,0.8863
8,16,keep top-k,0.6761
9,16,drop top-k,0.5990


,k,mode,acc
0,1,keep top-k,0.1863
1,1,drop top-k,0.9393
2,2,keep top-k,0.2231
3,2,drop top-k,0.9322
4,4,keep top-k,0.2723
5,4,drop top-k,0.9251
6,8,keep top-k,0.4145
7,8,drop top-k,0.8863
8,16,keep top-k,0.6761
9,16,drop top-k,0.5990


## Activation Galleries

In [19]:
@torch.no_grad()
def activation_gallery(factor_model, name, component_indices=None, topn=8, alias=None):
    plus, minus, down, sigma = factor_model.decompose()
    order = component_strengths(factor_model).argsort(descending=True)
    if component_indices is None:
        component_indices = list(range(min(6, sigma.numel())))
    acts = factor_model.component_activations(train.x)[:, order]
    fig = make_subplots(
        rows=len(component_indices),
        cols=topn + 2,
        column_titles=["pos", "neg"] + [f"ex {i+1}" for i in range(topn)],
        vertical_spacing=0.03,
        horizontal_spacing=0.01,
    )
    for row, comp in enumerate(component_indices, start=1):
        fig.add_heatmap(z=plus[:, comp].detach().cpu().view(28, 28).flip(0), colorscale="RdBu", zmid=0, showscale=False, row=row, col=1)
        fig.add_heatmap(z=minus[:, comp].detach().cpu().view(28, 28).flip(0), colorscale="RdBu", zmid=0, showscale=False, row=row, col=2)
        top = acts[:, comp].abs().topk(topn).indices
        for j, idx in enumerate(top.tolist(), start=3):
            fig.add_heatmap(z=train.x[idx].detach().cpu().view(28, 28).flip(0), colorscale="gray", showscale=False, row=row, col=j)
    fig.update_xaxes(visible=False).update_yaxes(visible=False)
    fig.update_layout(
        width=125 * (topn + 2),
        height=115 * len(component_indices),
        title=f"Activation galleries: {name}",
        template="plotly_white",
        margin=dict(l=5, r=5, t=45, b=5),
    )
    return save_fig(fig, f"activation_gallery_{name.replace(' ', '_').lower()}")

activation_gallery(best, best_name, topn=8)

## Seed Stability / Consensus

In [20]:
def top_patterns_for_matching(factor_model, k=16):
    plus, minus, down, sigma = factor_model.decompose()
    patterns = torch.cat([plus[:, :k], minus[:, :k]], dim=1)
    return normalize_columns(patterns)


@torch.no_grad()
def greedy_match_score(reference, candidate):
    sim = (reference.T @ candidate).abs().detach().cpu()
    used_ref, used_cand, vals = set(), set(), []
    flat_order = sim.flatten().argsort(descending=True).tolist()
    n_ref, n_cand = sim.shape
    for flat in flat_order:
        i, j = divmod(flat, n_cand)
        if i in used_ref or j in used_cand:
            continue
        used_ref.add(i)
        used_cand.add(j)
        vals.append(float(sim[i, j]))
        if len(vals) == min(n_ref, n_cand):
            break
    return sum(vals) / max(1, len(vals)), vals


def consensus_run(factory, reg, name, seeds=(11, 12, 13), steps=300, lr=0.035, k=16):
    models = []
    rows = []
    for seed in seeds:
        set_seed(seed)
        m, hist = fit_factor_model(factory(), f"{name} seed {seed}", steps=steps, lr=lr, reg=reg, seed=None, progress=True)
        models.append(m)
        rows.append(metric_dict(f"{name} seed {seed}", m))
    ref = top_patterns_for_matching(models[0], k=k)
    for seed, m, row in zip(seeds[1:], models[1:], rows[1:]):
        score, vals = greedy_match_score(ref, top_patterns_for_matching(m, k=k))
        row["consensus_vs_seed0"] = score
    rows[0]["consensus_vs_seed0"] = 1.0
    return models, rows

# Run this on the best-looking family. Reduce steps if iterating quickly.
consensus_models, consensus_rows = consensus_run(
    factory=lambda: EvidenceSplitModel(rank=BASE_RANK),
    reg=RegConfig(l1=2e-4, tv=2e-3, d_l1=2e-4, duplicate=1e-3),
    name="evidence split sparse smooth consensus",
    seeds=(11, 12, 13),
    steps=CONSENSUS_STEPS,
    lr=0.035,
    k=12,
)
show_table(consensus_rows)
plot_factor_components(consensus_models[0], "consensus reference evidence split sparse smooth", k=12)

evidence split sparse smooth consensus seed 11:   0%|          | 0/80 [00:00<?, ?it/s]

evidence split sparse smooth consensus seed 11 sim=-0.0006:   0%|          | 0/80 [00:00<?, ?it/s]

evidence split sparse smooth consensus seed 11 sim=0.0549:   0%|          | 0/80 [00:00<?, ?it/s] 

evidence split sparse smooth consensus seed 11 sim=0.3119:   0%|          | 0/80 [00:00<?, ?it/s]

evidence split sparse smooth consensus seed 11 sim=0.3119:   4%|▍         | 3/80 [00:00<00:02, 28.53it/s]

...

evidence split sparse smooth consensus seed 13 sim=0.8606:  94%|█████████▍| 75/80 [00:01<00:00, 41.65it/s]

evidence split sparse smooth consensus seed 13 sim=0.8606:  94%|█████████▍| 75/80 [00:01<00:00, 41.65it/s]

evidence split sparse smooth consensus seed 13 sim=0.8606: 100%|██████████| 80/80 [00:01<00:00, 41.84it/s]

evidence split sparse smooth consensus seed 13 sim=0.8606: 100%|██████████| 80/80 [00:01<00:00, 41.78it/s]

,name,similarity,test_acc,pattern_gini,locality_7x7,class_selectivity,top_sigma_frac,consensus_vs_seed0
0,evidence split sparse smooth consensus seed 11,0.859923,0.9399,0.433521,0.235469,0.113702,0.288248,1.000000
1,evidence split sparse smooth consensus seed 12,0.863464,0.9568,0.437121,0.245698,0.102267,0.281830,0.539269
2,evidence split sparse smooth consensus seed 13,0.860575,0.9536,0.435162,0.243239,0.100038,0.283096,0.540127


## Notes And Interpretation

Use this section after running the notebook. The intended claims to check are:

- Does the provided sparse baseline win reconstruction but lose visual clarity?
- Does forcing symmetry make components cleaner or merely underfit?
- Does total variation remove pixel-level artifacts without washing out digit strokes?
- Do class-sparse logits make components easier to name?
- Do evidence-split components make the positive/negative interpretation more honest?
- Are the best components stable across seeds, or are the nicest-looking examples optimizer artifacts?

A standout final answer should include both successes and failures. For example, it is totally acceptable if nonnegative strokes look beautiful but reconstruct worse; that is a real interpretability tradeoff.

In [21]:
# Final compact export table for the README.
if pd is not None:
    final_df = pd.DataFrame(rows).sort_values("similarity", ascending=False)
    final_df.to_csv("figures/decomposition_results.csv", index=False)
    display(final_df)
else:
    rows

,name,similarity,test_acc,pattern_gini,locality_7x7,class_selectivity,top_sigma_frac
5,evidence split sparse smooth r32,0.877034,0.9548,0.440257,0.254946,0.120680,0.306341
4,evidence split r32,0.872256,0.9611,0.434809,0.237860,0.109408,0.290891
1,cp r32 symmetry-soft,0.860983,0.9511,0.437123,0.238074,0.103718,0.299290
0,provided Sparse r32,0.858908,0.9492,0.440377,0.264228,0.106274,0.295426
7,eigen seeded symmetric sparse smooth r32,0.802525,0.9265,0.223442,0.151670,0.117094,0.290446
3,symmetric sparse smooth r32,0.787751,0.9316,0.222727,0.138692,0.108916,0.300652
2,symmetric r32,0.778788,0.8867,0.221623,0.139064,0.116112,0.287478
6,nonnegative strokes r48,0.087126,0.4322,0.010952,0.063500,0.103726,0.277420


## Provisional Conclusion Template

After running the notebook, replace this with the concrete result. A useful shape is:

> The plain CP-style sparse decomposition achieved the strongest raw reconstruction, but its top components mixed several digit strokes. The best sparse/smooth evidence-split or symmetric variant gave up a small amount of cosine similarity while producing more localized and class-selective components. Ablation curves showed that the top components were not merely decorative: keeping only the leading components preserved a meaningful fraction of accuracy, and dropping them damaged predictions. Seed consensus was the strictest test; components that survived it are the most credible candidates for real shared model features.